# FinRAG QLoRA Fine-Tuning\n\nThis notebook fine-tunes `Qwen/Qwen2.5-7B-Instruct` with LoRA adapters on FinQA-style financial QA examples. Use a CUDA GPU runtime.

In [ ]:
import torch\nassert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU before running.'\nprint(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

Clone the project from GitHub. Drive is still mounted above because the LoRA adapter is saved there.

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/pendyal1/Financial_QA_ChatBot.git"
BRANCH   = "integration/finrag-final"
PROJECT_DIR = "/content/Financial_QA_ChatBot"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR],
        check=True,
    )
    print("Cloned.")
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)
    print("Pulled latest.")

%cd $PROJECT_DIR

In [ ]:
!bash scripts/colab_setup.sh\n# If this still reports mixed CUDA versions, use Runtime > Disconnect and delete runtime, then reconnect.

In [ ]:
!PYTHONPATH=src python -m finrag.fine_tuning --limit 2000

In [ ]:
!PYTHONPATH=src python -m finrag.train_qlora \\\n  --model-name Qwen/Qwen2.5-7B-Instruct \\\n  --train-file data/fine_tuning/finqa_train.jsonl \\\n  --output-dir /content/drive/MyDrive/finrag-adapters/qwen2_5_7b_finqa_lora \\\n  --epochs 1 \\\n  --max-length 1536 \\\n  --batch-size 1 \\\n  --gradient-accumulation-steps 16 \\\n  --learning-rate 2e-4 \\\n  --lora-r 16 \\\n  --lora-alpha 32

In [ ]:
!PYTHONPATH=src python -m finrag.hf_adapter_answer \\\n  --adapter-path /content/drive/MyDrive/finrag-adapters/qwen2_5_7b_finqa_lora \\\n  --model-name Qwen/Qwen2.5-7B-Instruct \\\n  'What risks did Apple report related to supply chains?'

## Serve Qwen From The Colab GPU Runtime\n\nLocal Streamlit should run on your Mac CPU. Colab should only host the Qwen 2.5 7B generation endpoint. Copy the public tunnel URL from the last cell and paste it into the local Streamlit app.

In [ ]:
import subprocess, os, time

ADAPTER = "/content/drive/MyDrive/Generative AI Project FinRAG/finrag_lora_adapter"

# Kill any existing server instance
subprocess.run(["pkill", "-f", "finrag.qwen_server"], capture_output=True)

cmd = [
    "python", "-m", "finrag.qwen_server",
    "--model-name", "Qwen/Qwen2.5-7B-Instruct",
    "--port", "8000",
]
if os.path.isdir(ADAPTER):
    cmd += ["--adapter-path", ADAPTER]
    print(f"Adapter found — loading fine-tuned model.")
else:
    print(f"Adapter not found at:\n  {ADAPTER}\nLoading base model only.")

env = {**os.environ, "PYTHONPATH": "src"}
log = open("qwen_server.log", "w")
subprocess.Popen(cmd, env=env, stdout=log, stderr=log)

print("Server starting... waiting 10s before checking log.")
time.sleep(10)
with open("qwen_server.log") as f:
    print(f.read())

In [ ]:
import requests, json

# Smoke-test the /hallucinate endpoint.
# The NLI model (cross-encoder/nli-deberta-v3-small, ~450 MB) downloads on first call.
payload = {
    "answer": "Apple reported revenue of $391 billion in fiscal year 2024.",
    "passages": [
        {
            "chunk_id": "test-chunk-1",
            "score": 0.9,
            "ticker": "AAPL",
            "company": "Apple Inc.",
            "source": "10-K 2024",
            "source_url": "https://www.sec.gov/",
            "text": (
                "Net sales for fiscal 2024 totaled $391.0 billion, compared to "
                "$383.3 billion in fiscal 2023, an increase of 2 percent year over year."
            ),
        }
    ],
}

resp = requests.post("http://127.0.0.1:8000/hallucinate", json=payload, timeout=180)
resp.raise_for_status()
result = resp.json()
print("overall_risk   :", result["overall_risk"])
print("confidence     :", result["confidence_score"])
print("grounded_count :", result["grounded_count"])
print("claims:")
for c in result["claims"]:
    print(f"  [{c['label']}] {c['claim_text'][:80]}")

In [ ]:
# Force base model only, ignoring any saved adapter.\n# !pkill -f 'finrag.qwen_server' || true\n# !nohup env PYTHONPATH=src python -m finrag.qwen_server \\\n#   --model-name Qwen/Qwen2.5-7B-Instruct \\\n#   --port 8000 > qwen_server.log 2>&1 &

In [ ]:
%%bash\n# Expose the Colab Qwen server publicly so local Streamlit can call it.\npkill -f cloudflared || true\nwget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared\nchmod +x cloudflared\nnohup ./cloudflared tunnel --url http://127.0.0.1:8000 > cloudflared.log 2>&1 &\nsleep 8\ngrep -o 'https://[-a-zA-Z0-9.]*trycloudflare.com' cloudflared.log | tail -1

### Fallback: expose the Qwen server with ngrok\n\nUse this if Cloudflare Quick Tunnel returns a 500 error. You need a free ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken.

In [ ]:
from pyngrok import ngrok\n\nNGROK_AUTHTOKEN = ''  # paste your ngrok token here\nif not NGROK_AUTHTOKEN:\n    raise ValueError('Paste your ngrok auth token into NGROK_AUTHTOKEN')\n\nngrok.set_auth_token(NGROK_AUTHTOKEN)\npublic_url = ngrok.connect(8000, 'http').public_url\nprint(public_url)